# Random Location Run-of-River Example

This example selects one hydropower location, tries to extract its daily discharge, and then computes run-of-river hydropower using the hydro logic in RESKit.

In [1]:
# import base libraies
import pandas as pd
import numpy as np

# get current repo path
from pathlib import Path
repo_root = Path.cwd().resolve()
while not (repo_root / "pyproject.toml").exists() and repo_root.parent != repo_root:
    repo_root = repo_root.parent

prepare the plant placement table

In [ ]:
rng = np.random.default_rng(42)
# # 1. Pick one random location inside a rough Germany bounding box.
# random_lat = float(rng.uniform(47.0, 54.5))
# random_lon = float(rng.uniform(5.0, 15.5))

# 2. Or choose a specific hydropower plant location
hydro_name = 'Bergheim'
hydro_id = 'GHR03171'
hydro_lat = 48.750807
hydro_lon = 11.273237
hydro_capacity = 23_700.0  # 23.7 MW in kW
hydro_head = 6 # m

placements = pd.DataFrame(
    {
        "lat": [hydro_lat],
        "lon": [hydro_lon],
        "capacity": [hydro_capacity],  # capacity is stored in kW
    }
)

year = 2020
time_index = pd.date_range(f"{year}-01-01", f"{year}-12-31", freq="D")
net_head_m = hydro_head

placements

,lat,lon,capacity
0,48.750807,11.273237,23700.0


Discharge extraction option (1) Try a real ParFlow-based discharge extraction first, from the nearest neighbour to the plant. 
If that is not available in the local environment, fall back to a synthetic daily discharge series so the example remains runnable.

In [ ]:
from reskit.hydro.core.parflow_discharge_extraction import extract_selected_discharge_alluvium, retrieve_discharge_data

root_dir = repo_root / "reskit" / "hydro" / "_external_module" / "parflow_600m_runs"

try:
    discharge_m3_per_day = retrieve_discharge_data(
        year=year,
        latitudes=[hydro_lat],
        longitudes=[hydro_lon],
        root_dir=str(root_dir),
    )
    discharge_m3_per_day = np.asarray(discharge_m3_per_day, dtype=float)
    discharge_source = "ParFlow extraction"
except Exception as exc:
    import traceback

    traceback.print_exc()
    days = len(time_index)
    seasonal = 320.0 + 180.0 * np.sin(np.linspace(0.0, 2.0 * np.pi, days))
    noise = rng.normal(0.0, 35.0, size=days)
    discharge_m3_per_day = np.clip(seasonal + noise, 0.0, None)[None, :]
    discharge_source = f"Synthetic fallback: {type(exc).__name__}"

(discharge_m3_per_day.shape, discharge_source)

----------------------------------------
JSON record: 0
location ID, lon, lat, depth, sim-data: location_0, 11.273237, 48.750807, 0.2, https://service.tereno.net/thredds/dodsC/forecastnrw/products/ParFlow-DE06-HC_v03/sfd_DE05_ECMWF-HRES_hindcast_r1i1p2_FZJ-IBG3-ParFlowCLM380_hgfadapter-h00-v03bJuwelsGpuProdClimatologyTl_1day_20200101-20201231.nc
POI: index x-y and lon-lat on model grid: 1163, 653, 11.274261474609375, 48.75282287597656
ATTENTION: your chosen POI (point of interest) is located directly on a lake, river, or ocean grid element, it is recommended to specify an alternative, nearby longitude-latitude coordinate pair
indices of alternative close-by grid elements to POI:  [(652, 1163), (653, 1162), (652, 1162), (653, 1164), (652, 1164), (654, 1163), (654, 1162), (651, 1163), (654, 1164)]
alternative, recommended coordinate pairs close-by, not on river, lake, or ocean, Lon-Lat [dec deg], 5 digits, edit your JASON file accordingly:  11.26672 48.74685
alternative, recommended coor

((1, 366), 'ParFlow extraction')

Discharge extraction option (2) Try a real ParFlow-based discharge extraction but from the nearest alluvium grid cell to the plant

In [ ]:
root_dir = repo_root / "reskit" / "hydro" / "_external_module" / "parflow_600m_runs"
alluvium_mask_file = root_dir / "alluvium_mask.nc"
indicator_file = root_dir / "DE-0055_INDICATOR_regridded_rescaled_SoilGrids250-v2017_BGRvector_newAllv.nc"

try:
    selected_discharge_result = extract_selected_discharge_alluvium(
        year=year,
        plant_lats=placements["lat"].values,
        plant_lons=placements["lon"].values,
        root_dir=str(root_dir),
        alluvium_mask_file=str(alluvium_mask_file),
        indicator_file=str(indicator_file),
    )
    summary = selected_discharge_result["selected_discharge_m3_per_day"].shape, selected_discharge_result["selected_from_alluvium"]
except Exception as exc:
    summary = f"ParFlow extraction skipped: {type(exc).__name__}"

summary

----------------------------------------
JSON record: 0
location ID, lon, lat, depth, sim-data: location_0, 11.275018692016602, 48.747344970703125, 0.2, https://service.tereno.net/thredds/dodsC/forecastnrw/products/ParFlow-DE06-HC_v03/sfd_DE05_ECMWF-HRES_hindcast_r1i1p2_FZJ-IBG3-ParFlowCLM380_hgfadapter-h00-v03bJuwelsGpuProdClimatologyTl_1day_20200101-20201231.nc
POI: index x-y and lon-lat on model grid: 1163, 652, 11.275018692016602, 48.747344970703125
ATTENTION: your chosen POI (point of interest) is located directly on a lake, river, or ocean grid element, it is recommended to specify an alternative, nearby longitude-latitude coordinate pair
indices of alternative close-by grid elements to POI:  [(652, 1164), (652, 1162), (651, 1163), (653, 1163), (651, 1162), (651, 1164), (653, 1162), (653, 1164), (652, 1165)]
alternative, recommended coordinate pairs close-by, not on river, lake, or ocean, Lon-Lat [dec deg], 5 digits, edit your JASON file accordingly:  11.26672 48.74685
alternativ

((1, 366), array([ True]))

Hydropower calculation option (1) Calculate run-of-river power generation and capacity factor using HydroWorkflowManager (capped by capacity by default)

In [ ]:
from reskit.hydro.workflows.hydro_workflow_manager import HydroWorkflowManager

# Use HydroWorkflowManager directly instead of the convenience workflow
wf = HydroWorkflowManager(placements)

discharge_m3_per_day = np.asarray(discharge_m3_per_day) # or use selected_discharge_result["selected_discharge_m3_per_day"]

# Set the workflow time index
wf.set_time_index(time_index)

# Run the run-of-river calculation from daily discharge
wf.simulate_run_of_river_from_daily_discharge(
    discharge_m3_per_day=discharge_m3_per_day,
    net_head_m=net_head_m,
    efficiency=0.88,
)

# Convert to xarray dataset and show key outputs 
result = wf.to_xarray(output_variables=[
    "discharge_m3_per_day",
    "usable_discharge_m3_per_day",
    "capacity_factor",
    "total_system_generation",
])
result[["capacity_factor", "total_system_generation"]]

<xarray.Dataset> Size: 9kB
Dimensions:                  (time: 366, location: 1)
Coordinates:
  * time                     (time) datetime64[ns] 3kB 2020-01-01 ... 2020-12-31
  * location                 (location) int64 8B 0
Data variables:
    capacity_factor          (time, location) float64 3kB 0.9195 ... 0.8349
    total_system_generation  (time, location) float64 3kB 5.23e+05 ... 4.749e+05

Hydropower calculation option (2) Calculate run-of-river power generation and capacity factor using existing workflows, using daily discharge data from any source


In [ ]:
from reskit.hydro.workflows import run_of_river_daily_discharge_workflow

full_parflow_result = run_of_river_daily_discharge_workflow(
    placements=placements,
    discharge_m3_per_day=discharge_m3_per_day,
    time_index=time_index,
    net_head_m=net_head_m,
    efficiency=0.88,
    output_netcdf_path=None,
    output_variables=[
        "capacity_factor",
        "total_system_generation",
        "selected_candidate_idx",
        "selected_from_alluvium",
    ],
)

full_parflow_result[["capacity_factor", "total_system_generation"]]

<xarray.Dataset> Size: 9kB
Dimensions:                  (time: 366, location: 1)
Coordinates:
  * time                     (time) datetime64[ns] 3kB 2020-01-01 ... 2020-12-31
  * location                 (location) int64 8B 0
Data variables:
    capacity_factor          (time, location) float64 3kB 0.9195 ... 0.8349
    total_system_generation  (time, location) float64 3kB 5.23e+05 ... 4.749e+05

Hydropower calculation option (2) Calculate run-of-river power generation and capacity factor using existing workflows, using Parflow-based discharge extraction from the nearest alluvium grid cell to the plant.

In [10]:
# cap production by capacity
from reskit.hydro.workflows import run_of_river_parflow_alluvium_workflow

capped_production = run_of_river_parflow_alluvium_workflow(
    placements=placements,
    year=year,
    time_index=time_index,
    net_head_m=net_head_m,
    extraction_root_dir=str(root_dir),
    alluvium_mask_file=str(alluvium_mask_file),
    indicator_file=str(indicator_file),
    efficiency=0.88,
    cap_production_by_capacity=True,
    output_variables=[
        "capacity_factor",
        "total_system_generation",
        "selected_candidate_idx",
        "selected_from_alluvium",
    ],
)

full_parflow_result[["capacity_factor", "total_system_generation"]]

----------------------------------------
JSON record: 0
location ID, lon, lat, depth, sim-data: location_0, 11.275018692016602, 48.747344970703125, 0.2, https://service.tereno.net/thredds/dodsC/forecastnrw/products/ParFlow-DE06-HC_v03/sfd_DE05_ECMWF-HRES_hindcast_r1i1p2_FZJ-IBG3-ParFlowCLM380_hgfadapter-h00-v03bJuwelsGpuProdClimatologyTl_1day_20200101-20201231.nc
POI: index x-y and lon-lat on model grid: 1163, 652, 11.275018692016602, 48.747344970703125
ATTENTION: your chosen POI (point of interest) is located directly on a lake, river, or ocean grid element, it is recommended to specify an alternative, nearby longitude-latitude coordinate pair
indices of alternative close-by grid elements to POI:  [(652, 1164), (652, 1162), (651, 1163), (653, 1163), (651, 1162), (651, 1164), (653, 1162), (653, 1164), (652, 1165)]
alternative, recommended coordinate pairs close-by, not on river, lake, or ocean, Lon-Lat [dec deg], 5 digits, edit your JASON file accordingly:  11.26672 48.74685
alternativ

<xarray.Dataset> Size: 9kB
Dimensions:                  (time: 366, location: 1)
Coordinates:
  * time                     (time) datetime64[ns] 3kB 2020-01-01 ... 2020-12-31
  * location                 (location) int64 8B 0
Data variables:
    capacity_factor          (time, location) float64 3kB 0.9206 ... 0.8361
    total_system_generation  (time, location) float64 3kB 5.237e+05 ... 4.756...

In [11]:
# no cap on production
from reskit.hydro.workflows import run_of_river_parflow_alluvium_workflow

uncapped_production = run_of_river_parflow_alluvium_workflow(
    placements=placements,
    year=year,
    time_index=time_index,
    net_head_m=net_head_m,
    extraction_root_dir=str(root_dir),
    alluvium_mask_file=str(alluvium_mask_file),
    indicator_file=str(indicator_file),
    efficiency=0.88,
    cap_production_by_capacity=False,
    output_variables=[
        "capacity_factor",
        "total_system_generation",
        "selected_candidate_idx",
        "selected_from_alluvium",
    ],
)

full_parflow_result[["capacity_factor", "total_system_generation"]]

----------------------------------------
JSON record: 0
location ID, lon, lat, depth, sim-data: location_0, 11.275018692016602, 48.747344970703125, 0.2, https://service.tereno.net/thredds/dodsC/forecastnrw/products/ParFlow-DE06-HC_v03/sfd_DE05_ECMWF-HRES_hindcast_r1i1p2_FZJ-IBG3-ParFlowCLM380_hgfadapter-h00-v03bJuwelsGpuProdClimatologyTl_1day_20200101-20201231.nc
POI: index x-y and lon-lat on model grid: 1163, 652, 11.275018692016602, 48.747344970703125
ATTENTION: your chosen POI (point of interest) is located directly on a lake, river, or ocean grid element, it is recommended to specify an alternative, nearby longitude-latitude coordinate pair
indices of alternative close-by grid elements to POI:  [(652, 1164), (652, 1162), (651, 1163), (653, 1163), (651, 1162), (651, 1164), (653, 1162), (653, 1164), (652, 1165)]
alternative, recommended coordinate pairs close-by, not on river, lake, or ocean, Lon-Lat [dec deg], 5 digits, edit your JASON file accordingly:  11.26672 48.74685
alternativ

<xarray.Dataset> Size: 9kB
Dimensions:                  (time: 366, location: 1)
Coordinates:
  * time                     (time) datetime64[ns] 3kB 2020-01-01 ... 2020-12-31
  * location                 (location) int64 8B 0
Data variables:
    capacity_factor          (time, location) float64 3kB 0.9206 ... 0.8361
    total_system_generation  (time, location) float64 3kB 5.237e+05 ... 4.756...

In [12]:
# Compute differences (uncapped - capped)
diff_generation = uncapped_production['total_system_generation'] - capped_production['total_system_generation']
diff_cf = uncapped_production['capacity_factor'] - capped_production['capacity_factor']

print('Summary: total_system_generation (uncapped - capped)')
print(diff_generation.to_dataframe().describe())
print('\nSummary: capacity_factor (uncapped - capped)')
print(diff_cf.to_dataframe().describe())

# Return the two difference arrays for inspection
diff_generation, diff_cf

Summary: total_system_generation (uncapped - capped)
       total_system_generation
count             3.660000e+02
mean              4.373378e+04
std               1.760282e+05
min               0.000000e+00
25%               0.000000e+00
50%               0.000000e+00
75%               0.000000e+00
max               1.414533e+06

Summary: capacity_factor (uncapped - capped)
       capacity_factor
count       366.000000
mean          0.076888
std           0.309473
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max           2.486872


(<xarray.DataArray 'total_system_generation' (time: 366, location: 1)> Size: 3kB
 array([[0.00000000e+00],
        [0.00000000e+00],
        [0.00000000e+00],
        [0.00000000e+00],
        [0.00000000e+00],
        [0.00000000e+00],
        [0.00000000e+00],
        [0.00000000e+00],
        [0.00000000e+00],
        [0.00000000e+00],
        [0.00000000e+00],
        [0.00000000e+00],
        [0.00000000e+00],
        [0.00000000e+00],
        [0.00000000e+00],
        [0.00000000e+00],
        [0.00000000e+00],
        [0.00000000e+00],
        [0.00000000e+00],
        [0.00000000e+00],
 ...
        [0.00000000e+00],
        [0.00000000e+00],
        [0.00000000e+00],
        [0.00000000e+00],
        [0.00000000e+00],
        [0.00000000e+00],
        [0.00000000e+00],
        [0.00000000e+00],
        [0.00000000e+00],
        [0.00000000e+00],
        [0.00000000e+00],
        [0.00000000e+00],
        [0.00000000e+00],
        [8.34002129e+04],
        [2.64622943e+05],
    